# Generate the Distance Matrices 
Gemerate a matrix for each metric (upos, ufeats, uas, las)

The metrics are available at the folder `metrics`
```
metrics
├── af_afribooms-on-af_afribooms-ud-test.json
├── af_afribooms-on-de_gsd-ud-test.json
├── af_afribooms-on-el_gud-ud-test.json
├── af_afribooms-on-it_isdt-ud-test.json
├── de_gsd-on-af_afribooms-ud-test.json
├── de_gsd-on-de_gsd-ud-test.json
├── de_gsd-on-el_gud-ud-test.json
├── de_gsd-on-it_isdt-ud-test.json
├── el_gud-on-af_afribooms-ud-test.json
├── el_gud-on-de_gsd-ud-test.json
├── el_gud-on-el_gud-ud-test.json
├── el_gud-on-it_isdt-ud-test.json
├── it_isdt-on-af_afribooms-ud-test.json
├── it_isdt-on-de_gsd-ud-test.json
├── it_isdt-on-el_gud-ud-test.json
└── it_isdt-on-it_isdt-ud-test.json
```

In [ ]:
import os
import json
import pandas as pd

In [ ]:
def get_languages(folder_path='metrics'):
    """
    Parses the metrics folder to extract a unique, sorted list of 
    all source and target languages found in the filenames.
    """
    languages = set()
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):
            # Remove suffix and split by '-on-'
            name_part = filename.replace('-ud-test.json', '')
            parts = name_part.split('-on-')
            if len(parts) == 2:
                languages.add(parts[0])
                languages.add(parts[1])
    
    # Return sorted list to ensure consistent matrix order
    return sorted(list(languages))

In [ ]:
def create_single_metric_matrix(metric_name, folder_path='metrics'):
    languages = get_languages(folder_path)
    
    # Initialize the empty DataFrame with the extracted languages
    matrix = pd.DataFrame(index=languages, columns=languages)
    
    # Iterate through files and fill the DataFrame
    for filename in os.listdir(folder_path):
        if filename.endswith('.json'):
            name_part = filename.replace('-ud-test.json', '')
            source, target = name_part.split('-on-')
            
            with open(os.path.join(folder_path, filename), 'r') as f:
                data = json.load(f)
                
                if metric_name in data:
                    matrix.loc[source, target] = data[metric_name]
    
    return matrix

In [ ]:
upos_df = create_single_metric_matrix('upos')
print("--- UPOS Matrix ---")
print(upos_df)

In [ ]:
ufeats_df = create_single_metric_matrix('ufeats')
print("--- UFeats Matrix ---")
print(ufeats_df)

In [ ]:
uas_df = create_single_metric_matrix("uas")
print("--- UAS Matrix ---")
print(uas_df)

In [ ]:
las_df = create_single_metric_matrix("las")
print("--- LAS Matrix ---")
print(las_df)

Plot the matrices as heatmaps

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
def plot_heatmap(matrix, metric_name, filename):
    """
    Plots a heatmap for the given matrix and saves it as a PNG file.
    """
    plt.figure(figsize=(8, 8))
    # We ensure the matrix is float type for plotting
    sns.heatmap(matrix.astype(float), annot=True, cmap='viridis', fmt='.2f')
    plt.title(f'Heatmap for {metric_name.upper()}')
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()

In [ ]:
plot_heatmap(upos_df, "upos", "upos_scores_heatmap.png")

In [ ]:
plot_heatmap(ufeats_df, "ufeats", "ufeats_scores_heatmap.png")

In [ ]:
plot_heatmap(uas_df, "uas", "uas_scores_heatmap.png")

In [ ]:
plot_heatmap(las_df, "las", "las_scores_heatmap.png")

## Convert Scores to Distances

The diagonal elements should be 0 (the distance of a language from itself should be 0)


- Normalize each row by the diagonal value. For each row, divide the elements of the row by the element in the diagonal. That way we normalise the scores (percentages) to represent similarities (ranging from 0.0 to 1.0)
$$ R_{ij} = \frac{S_{ij}}{S_{jj}} $$
This yields numbers ranging for 0.0 to 1.0 and the diagonal is 1.0 (the similarity of a language to itself is one).
- We subtract the normalized value from 1.0 to go from similarities to distances. The distance of a language from itself becomes 0
$$ D_{ij} = 1.0 - R_{ij} $$
- We then make the matrix symmetrical 
$$ D_{ij}^{\text{sym}} = \frac{D_{ij} + D_{ji}}{2.0} $$
Where ${D_{ij}}$ is the original matrix and $D_{ji}$ is the transpose of the matrix.
 


In [ ]:
import numpy as np


def ud_scores_to_distance(S, self_normalize=True):
  """S: n x n matrix (dataframe) of raw parsing scores (e.g., LAS percentages)

  rows = source (model), cols = target (test set)
  """

  # Save the index and columns if S is a DataFrame
  if isinstance(S, pd.DataFrame):
    index_names = S.index
    col_names = S.columns
  else:
    index_names = None
    col_names = None

  S = np.array(S, dtype=float)
  n = S.shape[0]

  if self_normalize:
    # Normalize by target self-performance to control for language difficulty
    diag = np.diagonal(S)
    # S_ij relative to how well the target language parses itself
    # Dividing matrix S (shape n×n) by this column vector divides every score in row i 
    # by the self-parsing score of language i (S_ii). This creates ratios between 0 and 1
    Normalized = S / diag[:, np.newaxis]
    # Convert to distance: higher relative performance = smaller distance
    D = 1.0 - Normalized
  else:
    # Simple linear scale assuming 0-100 range
    D = (100.0 - S) / 100.0

  # Symmetrize the distance matrix
  D_sym = (D + D.T) / 2.0

  # Force exact zeros on the diagonal
  # the diagonal should already be 0 from the normalization step but
  # this ensures that if there is an overflow/underflow from the division
  # the diagonal will be a true zero value (instead of a very small number)
  # We don't strictly need it but it guarantees your matrix is mathematically 
  # pristine before we pass it into downstream algorithms.
  np.fill_diagonal(D_sym, 0.0)

  # Convert back to a DataFrame with original names if they existed
  if index_names is not None and col_names is not None:
    return pd.DataFrame(D_sym, index=index_names, columns=col_names)

  return D_sym

In [ ]:
D_upos = ud_scores_to_distance(upos_df)

In [ ]:
D_upos.to_excel("upos_distances.xlsx")

In [ ]:
plot_heatmap(D_upos, "upos", "upos_distance_heatmap.png")

In [ ]:
D_ufeats = ud_scores_to_distance(ufeats_df)

In [ ]:
D_ufeats.to_excel("ufeats_distances.xlsx")

In [ ]:
plot_heatmap(D_ufeats, "ufeats", "ufeats_distance_heatmap.png")

In [ ]:
D_uas = ud_scores_to_distance(uas_df)

In [ ]:
D_uas.to_excel("uas_distances.xlsx")

In [ ]:
plot_heatmap(D_uas, "uas", "uas_distance_heatmap.png")

In [ ]:
D_las = ud_scores_to_distance(las_df)

In [ ]:
D_las.to_excel("ulas_distances.xlsx")

In [ ]:
plot_heatmap(D_las,"las", "las_distance_heatmap.png")